In [2]:
%load_ext autoreload
%autoreload 2
import os, sys, re
import numpy as np
import pandas as pd
import scanpy as sc
import torch
import time
import PINN
from PINN import reader, models, pl, tl
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from torchdiffeq import odeint

import matplotlib as mpl
import seaborn as sns
from matplotlib.patches import Patch


os.chdir("/ssd/users/Wergillius/Project/PINN_dynamics")

In [3]:
import palantir
import cellrank as cr

In [4]:
sc.settings.set_figure_params(frameon=False, dpi=50, figsize=(3,3))

# Exp configs

In [5]:
# find all config files
config_dir = "logs/ery_mk_Aug1_multiscaled/pde_params_tsense/"
configs = [file for file in os.listdir(config_dir) if file.endswith('.json')]

In [6]:
# load config for different runs
config_dict = {}
for js in configs:
    version = js.split("_")[0][1:] # after V
    config_dict[version] = PINN.ExperimentConfig(os.path.join(config_dir, js))

In [7]:
for v,config in config_dict.items():
    print(v)
    print(config.find_lastest_ckpt())
    print("/n")

1
/ssd/users/Wergillius/Project/PINN_dynamics/logs/ery_mk_Aug1_multiscaled/pde_params_tsense/lightning_logs/version_1/checkpoints/epoch=211-val_loss=0.92638135.ckpt
/n
0
/ssd/users/Wergillius/Project/PINN_dynamics/logs/ery_mk_Aug1_multiscaled/pde_params_tsense/lightning_logs/version_0/checkpoints/epoch=204-val_loss=0.90176910.ckpt
/n


# load model and dataset

In [8]:
config.dataset_config

{'cellstate_key': 'DM_EigenVectors_multiscaled',
 'n_grid': 300,
 'n_dimension': 3,
 'kde_kws': {'bw_method': None},
 'timepoint_idx': [0, 1, 2, 3, 4, 6, 8],
 'deltax_key': 'delta_DM_multiscaled',
 'norm_time': 'True',
 'knn_volume': 'False'}

In [21]:
# # load dataset
dataset_name = config.experiment_config['dataset']
print(dataset_name)
adata = sc.read_h5ad(f'data/{dataset_name}.h5ad')

ery_mk_Aug1


In [10]:
for dmkey in ['delta_DM', 'delta_DM_multiscaled', 'delta_DM_scaled']:
    print(adata.obsm[dmkey].shape)

(22740, 5)
(22740, 3)
(22740, 5)


In [11]:
ds_config = config.dataset_config.copy()
ds_config['timepoint_idx'] = None

full_DS = PINN.reader.TwoTimpepoint_AnnDS(adata,split=None,**ds_config)

# train_DS = PINN.reader.TwoTimpepoint_AnnDS(adata,split='train',**ds_config)
# val_DS = PINN.reader.TwoTimpepoint_AnnDS(adata,split='val',**ds_config)
# test_DS = PINN.reader.TwoTimpepoint_AnnDS(adata,split='test',**ds_config)


Dataset : Computing density :
	 `density_funs` not specified, default estimator gaussian kde
Dataset : Use KNN distances to compute the volume and rescale density
Dataset : smoothing KNN derived single-cell volume..
Dataset : all cells are used


In [12]:
# load models
model_dict = {}
for v in config_dict:
    ckpt = config_dict[v].find_lastest_ckpt()
    model_dict[v] = PINN.models.pde_params.load_from_checkpoint(ckpt, map_location='cpu')

# scvelo

In [13]:
from PINN.functions import reader_funs
import scvelo as scv

In [14]:
cellstate_key = config.dataset_config['cellstate_key']
pde_model = model_dict[v]

In [ ]:
# # not needed for py
# for suffix in ["", "_scaled","_multiscaled"]:
#     vkey = f"DM_EigenVectors{suffix}"

#     cellstate_ad = PINN.tl.make_coord_adata(adata, cellstate_key=f"DM_EigenVectors{suffix}", n_dimension=5)
#     # sc.pp.neighbors(cellstate_ad, n_neighbors=15)

#     cellstate_ad.layers[vkey] = adata.obsm[f'delta_DM{suffix}'].copy()


#     # compute velocity graph
#     scv.tl.velocity_graph(cellstate_ad,  vkey=vkey, xkey='cellstate', n_jobs=20)

#     # vis
#     fig_velocity = plt.figure(dpi=50, figsize=(4,4))
#     ax = fig_velocity.gca()
#     scv.pl.velocity_embedding_stream(cellstate_ad, color='anno_man', vkey=vkey, 
#                                     basis='umap', ax=ax, 
#                                     legend_loc='right', alpha=0.01,
#                                     title=vkey, )
#                                     # save=f"{result_dir}/{vkey}_velo.png")

In [25]:
cellstate_key = config_dict[v].dataset_config['cellstate_key']
cellstate_key

'DM_EigenVectors_multiscaled'

In [26]:
v_dict = {}
for v, pde_model in model_dict.items():
    g_pred_ay, v_pred_ay, D_pred_ay = pde_model.predict_param(full_DS)

    result_dir = config_dict[v].experiment_config['checkpoint_dir'].replace("logs","results").replace("lightning_logs/","")
    PINN.tl.make_dir(result_dir)
    print(result_dir)

    fig_g,axs = PINN.pl.params_in_umap(adata, g_pred_ay, param='g')
    fig_g.savefig(os.path.join(result_dir,'g.png'), transparent=True)


    if D_pred_ay.shape[-1] == 1:
        fig_D,axs = PINN.pl.params_in_umap(adata, D_pred_ay.squeeze(), param='D')
        fig_D.savefig(os.path.join(result_dir,'D.png'), transparent=True)


    v_pred_norm = np.sqrt(np.sum(v_pred_ay**2, axis=-1))
    fig_v,axs = PINN.pl.params_in_umap(adata, v_pred_norm, param='v')
    fig_v.savefig(os.path.join(result_dir,'v.png'), transparent=True)


    # visualize like RNA velocity
    cellstate_ad = PINN.tl.make_coord_adata(adata, cellstate_key=cellstate_key, n_dimension=8, v = v_pred_ay)
    vkeys = [k for k in list(cellstate_ad.layers.keys()) if k.endswith("v")]

    for vkey in vkeys:

        # compute velocity graph
        scv.tl.velocity_graph(cellstate_ad,  vkey=vkey, xkey='cellstate', n_jobs=20)

        # vis
        fig_velocity = plt.figure(dpi=100, figsize=(4,4))
        ax = fig_velocity.gca()
        scv.pl.velocity_embedding_stream(cellstate_ad, color='anno_man', vkey=vkey, 
                                        basis='umap', ax=ax, 
                                        legend_loc='right', alpha=0.01,
                                        title=vkey, 
                                        save=f"{result_dir}/{vkey}_velo.png")

# simulate density and then evaluate

In [27]:
v = '0'
performance_js = []
timepoints = adata.uns['pop']['t']
n_timepoints = timepoints.shape[0]

for v in model_dict:
    device = 'cuda:4'
    pde_model = model_dict[v].to(device).eval()
    u_b = full_DS.u_b.cpu().numpy()

    u_sim = PINN.tl.density_shortterm_simulation(pde_model, DataSet=full_DS, timepoints=adata.uns['pop']['t'])
    u_int_all = np.concatenate([u_b[0,None], u_sim], axis=0)

    KLD_ls = PINN.tl.KLD_density(u_b, u_int_all)
    W1 = PINN.tl.W_distance(u_b, u_int_all, p=1)
    W2 = PINN.tl.W_distance(u_b, u_int_all)

    df = pd.DataFrame({
        "v":[v]*n_timepoints, 'KLD':KLD_ls,
         'W1': W1, 'W2':W2,
         't':timepoints
        })
    performance_js.append(df)

# concat all df
density_performance_df = pd.concat(performance_js, axis=0)

0it [00:00, ?it/s]

simulating from timepoint 3 to 7
simulating from timepoint 7 to 12
simulating from timepoint 12 to 27
simulating from timepoint 27 to 49
simulating from timepoint 49 to 76
simulating from timepoint 76 to 112
simulating from timepoint 112 to 161
simulating from timepoint 161 to 269


0it [00:00, ?it/s]

simulating from timepoint 3 to 7
simulating from timepoint 7 to 12
simulating from timepoint 12 to 27
simulating from timepoint 27 to 49
simulating from timepoint 49 to 76
simulating from timepoint 76 to 112
simulating from timepoint 112 to 161
simulating from timepoint 161 to 269


In [ ]:
density_performance_df.to_csv("logs/ery_mk_TM1/pde_params_tsense/density_perfomance.csv")

In [37]:
config.experiment_config['version']

0

In [38]:
result_dir = config.experiment_config['save_dir'].replace("logs", "results")
version = config.experiment_config['version']
result_dir = result_dir + "_0"

In [39]:
config.result_dir = result_dir

In [ ]:
config.dataset_config['n_dimension']

{'cellstate_key': 'DM_EigenVectors_multiscaled',
 'n_grid': 300,
 'n_dimension': 3,
 'kde_kws': {'bw_method': None},
 'timepoint_idx': [0, 1, 2, 3, 4, 6, 8],
 'deltax_key': 'delta_DM_multiscaled',
 'norm_time': 'True',
 'knn_volume': 'False'}

# simulate trajectory for HSC

In [17]:
HSC_ad = adata[adata.obs['anno_man']=='HSC'].copy()

init_s_by_time = {}

for t in HSC_ad.uns['pop']['t']:

    ad_t = HSC_ad[HSC_ad.obs['timepoint_tx_days']==t]
    s_init_t = ad_t.obsm[cellstate_key].copy()

    init_s_by_time[str(int(t))] = s_init_t

In [18]:
timepoint_tx_days = sorted(adata.obs.timepoint_tx_days.unique())
t0 = timepoint_tx_days[0]

In [ ]:
v = '0'
device = 'cuda:7'
n_interval = 9

simulated_trajactory = {}

pde_model = model_dict[v].to(device).eval()

DT = PINN.models.Density_Transfer(pde_model)

for t,s0 in init_s_by_time.items():
    t = int(t)
    integration_time = np.linspace(t/t0, (t+15)/t0 ,n_interval+1) / pde_model.time_scale_factor

    s_traj_t = DT.cellstate_drift(s0, integration_time)

    simulated_trajactory[str(t)] = s_traj_t

    # assign cell type
    nn_annotation = PINN.tl.assign_nearest_cell(s_traj_t[-1], adata, cellstate_key, n_dimension=8, annotation='anno_man')

In [ ]:
# Calculate the proportion of different labels for each time point (column)
label_proportions = nn_annotation.apply(lambda col: col.value_counts(normalize=True), axis=0)

# Transpose to have time points as rows and cell types as columns
# Fill NaN values with 0 (for cases where a cell type doesn't appear in a time point)
label_proportions = label_proportions.T.fillna(0)

print(label_proportions)

           HSC  Int prog       Meg
0     0.000000  0.000000  1.000000
1     0.000000  0.416667  0.583333
2     0.000000  0.000000  1.000000
3     0.000000  0.083333  0.916667
4     0.250000  0.333333  0.416667
...        ...       ...       ...
1165  0.000000  0.416667  0.583333
1166  0.000000  0.416667  0.583333
1167  0.000000  0.416667  0.583333
1168  0.000000  0.500000  0.500000
1169  0.333333  0.333333  0.333333

[1170 rows x 3 columns]


In [ ]:
cellstate_key = config.dataset_config['cellstate_key']
timepoint_key = 'timepoint_tx_days'

t_list = train_DS.T_b

# Dynamic density transport

In [ ]:
# go for all HSC cells
from torch import nn
import gc
import torch.nn.functional as F
from PINN.models import Density_Transfer
from tqdm.auto import tqdm

# initial cell
n_interval = 10
ncell = 100

pde_model = pde_model.to(device)
DT = Density_Transfer(pde_model)

NameError: name 'pde_model' is not defined

In [ ]:
# # load dataset
dataset = config.experiment_config['dataset']
adata = sc.read_h5ad(f'data/{dataset}.h5ad')

ds_config = config.dataset_config.copy()
ds_config['timepoint_idx'] = None
full_DS = PINN.reader.TwoTimpepoint_AnnDS(adata,split=None,**ds_config)

In [45]:
config.raw_args.keys()

dict_keys(['config', 'dataset', 'cellstate_key', 'model', 'pretrained', 'gpu_devices', 'log_name', 'lr', 'schedule_lr', 'n_grid', 'n_dimension', 'timepoint_idx', 'knn_volume', 'batch_size', 'bw', 'tol', 'channels', 'D_penalty', 'deltax_key', 'deltax_weight', 'weight_intensity', 'time_scale_factor', 'norm_time', 'time_sensitive', 'progress_bar'])

In [ ]:


timepoint_tx_days = sorted(adata.obs.timepoint_tx_days.unique())
t0 = timepoint_tx_days[0]

Tmaps = {}
Tmaps_norm = {}
S_traj_lookup = {}
HSC_cbs = []

for it,t in tqdm(enumerate(timepoint_tx_days[:-1])):
    # if it==0:
    #     torch.cuda.memory._record_memory_history()

    integrate_time = np.linspace(t/t0, timepoint_tx_days[it+1]/t0 ,n_interval+1) / pde_model.time_scale_factor
    print(integrate_time)

    # find cell of time
    start_cell = adata.obs.query("`anno_man` == 'HSC' & `timepoint_tx_days` == @t").index
    start_cell = list(start_cell)
    HSC_cbs.append(start_cell)

    # define initital density and cellstates
    cell_index = [np.where(adata.obs_names == x)[0].item() for x in start_cell]
    u0 = Dataset.u_b[it, cell_index].float().to(device)
    s0 = torch.from_numpy(Dataset.cellstate[cell_index]).float().to(device)

    S_trajectory = DT.cellstate_drift(s0, integrate_time)
    Tmaps_t, Tmaps_t_norm = DT.transition_by_batch(s0, u0, integrate_time, n_interval=n_interval, ncell=ncell)

    del u0, s0
    
    S_traj_lookup[str(t)] = S_trajectory
    Tmaps_norm[str(t)] = Tmaps_t_norm
    Tmaps[str(t)] = Tmaps_t